# Using Pretrained CV Models

## Overview

Pretrained models from torchvision and HuggingFace provide strong feature extractors and task-specific heads trained on large datasets. Using them correctly is the single highest-leverage technique for applied computer vision with limited data.

**Available pretrained models (torchvision):**

| Family | Strengths | Params |
|---|---|---|
| ResNet18/50 | Robust baseline, well understood | 11M / 25M |
| EfficientNet-B0–B7 | Best accuracy/efficiency tradeoff | 5–66M |
| MobileNetV3 | Mobile/edge deployment | 5M |
| ViT-B/16 | Long-range spatial attention | 86M |
| Swin Transformer | Hierarchical windows | 28–88M |

**Key principle:** always benchmark the smallest model that meets accuracy requirements — inference cost, memory, and latency matter in production.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from copy import deepcopy

torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def make_patch(label, size=224, seed=0):
    r = np.random.default_rng(seed)
    palettes = {0: ([40,80],[100,160],[30,70]),
                1: ([80,130],[90,140],[40,80]),
                2: ([120,180],[70,110],[30,60])}
    lo, hi = zip(*[palettes[label][c] for c in range(3)])
    img = np.stack([r.integers(lo[c], hi[c], (size,size)) for c in range(3)], axis=2)
    noise = r.integers(-20, 20, img.shape)
    return Image.fromarray(np.clip(img + noise, 0, 255).astype(np.uint8))

N_CLASSES = 3
all_imgs   = [make_patch(lbl, seed=i+lbl*500) for lbl in range(N_CLASSES) for i in range(80)]
all_labels = [lbl for lbl in range(N_CLASSES) for _ in range(80)]
split_tr, split_va = int(0.7*len(all_imgs)), int(0.85*len(all_imgs))

class PatchDataset(Dataset):
    def __init__(self, imgs, labels, transform):
        self.imgs, self.labels, self.transform = imgs, labels, transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        return self.transform(self.imgs[i]), torch.tensor(self.labels[i], dtype=torch.long)

train_tfm = T.Compose([T.Resize((64,64)), T.RandomHorizontalFlip(),
    T.ColorJitter(0.3,0.3,0.2), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tfm  = T.Compose([T.Resize((64,64)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
train_dl = DataLoader(PatchDataset(all_imgs[:split_tr], all_labels[:split_tr], train_tfm), 32, shuffle=True)
val_dl   = DataLoader(PatchDataset(all_imgs[split_tr:split_va], all_labels[split_tr:split_va], eval_tfm), 32)
test_dl  = DataLoader(PatchDataset(all_imgs[split_va:], all_labels[split_va:], eval_tfm), 32)
print(f"Train={split_tr}, Val={split_va-split_tr}, Test={len(all_imgs)-split_va}")

---
## Loading and Adapting Pretrained Models

In [ ]:
try:
    import torchvision.models as models

    def adapt_model(arch, n_classes, freeze_backbone=True):
        if arch == 'resnet18':
            m = models.resnet18(weights='DEFAULT')
            in_f = m.fc.in_features
            m.fc = nn.Linear(in_f, n_classes)
            if freeze_backbone:
                for name, p in m.named_parameters():
                    if 'fc' not in name: p.requires_grad = False
        elif arch == 'efficientnet_b0':
            m = models.efficientnet_b0(weights='DEFAULT')
            in_f = m.classifier[1].in_features
            m.classifier[1] = nn.Linear(in_f, n_classes)
            if freeze_backbone:
                for name, p in m.named_parameters():
                    if 'classifier' not in name: p.requires_grad = False
        elif arch == 'mobilenet_v3_small':
            m = models.mobilenet_v3_small(weights='DEFAULT')
            in_f = m.classifier[3].in_features
            m.classifier[3] = nn.Linear(in_f, n_classes)
            if freeze_backbone:
                for name, p in m.named_parameters():
                    if 'classifier' not in name: p.requires_grad = False
        total     = sum(p.numel() for p in m.parameters())
        trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
        print(f"  {arch:22s}: {total:8,} total, {trainable:6,} trainable")
        return m

    print("Model sizes and trainable params (head-only mode):")
    models_dict = {}
    for arch in ['resnet18', 'efficientnet_b0', 'mobilenet_v3_small']:
        models_dict[arch] = adapt_model(arch, N_CLASSES, freeze_backbone=True)
except ImportError:
    print("pip install torchvision")

---
## Comparing Architectures

In [ ]:
try:
    import torchvision.models as models
    import time

    def train_and_eval(model, epochs=25, lr=1e-3):
        model = model.to(device)
        opt   = optim.AdamW([p for p in model.parameters() if p.requires_grad],
                             lr=lr, weight_decay=1e-4)
        crit  = nn.CrossEntropyLoss()
        sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        best_acc, best_state = 0, None
        for ep in range(epochs):
            model.train()
            for Xb, yb in train_dl:
                Xb, yb = Xb.to(device), yb.to(device)
                opt.zero_grad(); crit(model(Xb), yb).backward(); opt.step()
            sched.step()
            model.eval()
            with torch.no_grad():
                acc = np.mean([(model(Xb.to(device)).argmax(1)==yb.to(device)).float().mean().item()
                               for Xb, yb in val_dl])
            if acc > best_acc:
                best_acc = acc; best_state = deepcopy(model.state_dict())
        model.load_state_dict(best_state)
        # Measure inference latency
        model.eval()
        dummy = torch.zeros(1,3,64,64).to(device)
        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(100): model(dummy)
        latency_ms = (time.perf_counter()-t0)/100*1000
        return best_acc, latency_ms

    print(f"{'Model':24s} {'Val Acc':>9} {'Latency ms':>12}")
    for arch, m in models_dict.items():
        acc, lat = train_and_eval(m)
        print(f"  {arch:22s} {acc:9.3f} {lat:12.2f}")
except (ImportError, NameError):
    print("torchvision required for this demo")
    print("Expected pattern: EfficientNet > ResNet18 > MobileNet on accuracy")
    print("                  MobileNet > ResNet18 > EfficientNet on latency")

---
## Feature Extraction: Using Backbone as Embedder

In [ ]:
try:
    import torchvision.models as models
    from sklearn.decomposition import PCA
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score

    # Use pretrained ResNet18 as fixed feature extractor
    resnet = models.resnet18(weights='DEFAULT')
    # Remove final FC layer -> outputs 512-dim feature vectors
    backbone = nn.Sequential(*list(resnet.children())[:-1])
    backbone = backbone.to(device).eval()

    def extract_features(dl):
        feats, lbls = [], []
        with torch.no_grad():
            for Xb, yb in dl:
                f = backbone(Xb.to(device)).squeeze(-1).squeeze(-1)
                feats.append(f.cpu().numpy()); lbls.append(yb.numpy())
        return np.vstack(feats), np.concatenate(lbls)

    X_tr_feat, y_tr_feat = extract_features(train_dl)
    X_te_feat, y_te_feat = extract_features(test_dl)
    print(f"Feature shape: {X_tr_feat.shape}")

    # PCA visualisation
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X_tr_feat)
    fig, ax = plt.subplots(figsize=(7,5))
    colors = ['steelblue','#e74c3c','#2ecc71']
    class_names = ['Healthy','Moderate','Degraded']
    for cls in range(N_CLASSES):
        mask = y_tr_feat == cls
        ax.scatter(X_2d[mask,0], X_2d[mask,1], c=colors[cls],
                   label=class_names[cls], alpha=0.6, s=30)
    ax.set_title('ResNet18 Features (PCA 2D)')
    ax.legend(); plt.tight_layout(); plt.show()

    # Logistic regression on top of features
    lr_clf = LogisticRegression(max_iter=500, random_state=42)
    lr_clf.fit(X_tr_feat, y_tr_feat)
    acc = accuracy_score(y_te_feat, lr_clf.predict(X_te_feat))
    print(f"LR on ResNet18 features: {acc:.3f}")
except (ImportError, NameError):
    print("pip install torchvision, sklearn")

In [ ]:
# Model efficiency summary and selection guide
efficiency_data = {
    'Model':      ['ResNet18','ResNet50','EfficientNet-B0','EfficientNet-B3',
                   'MobileNetV3-S','ViT-B/16'],
    'Params (M)': [11.7, 25.6, 5.3, 12.2, 2.5, 86.6],
    'ImageNet Acc (%)': [69.8, 76.1, 77.7, 82.0, 67.7, 81.1],
    'Relative Speed': [1.0, 0.55, 1.2, 0.7, 2.5, 0.3],
}
import pandas as pd
df = pd.DataFrame(efficiency_data)
print("Pretrained model efficiency comparison:")
print(df.to_string(index=False))
fig, ax = plt.subplots(figsize=(9,5))
colors_m = ['steelblue','steelblue','#e74c3c','#e74c3c','#2ecc71','#9b59b6']
sc = ax.scatter(df['Params (M)'], df['ImageNet Acc (%)'],
                s=[r*200 for r in df['Relative Speed']],
                c=colors_m, alpha=0.7, edgecolors='white', linewidths=1.5)
for _, row in df.iterrows():
    ax.annotate(row['Model'], (row['Params (M)']+0.3, row['ImageNet Acc (%)']),
                fontsize=9)
ax.set_xlabel('Parameters (M)'); ax.set_ylabel('ImageNet Top-1 Acc (%)')
ax.set_title('Model Accuracy vs Size (bubble size = relative speed)')
plt.tight_layout(); plt.show()

---

## Common Pitfalls

**1. Using `weights=None` (random init) instead of `weights="DEFAULT"`**  
The entire benefit of pretrained models comes from the pretrained weights. `models.resnet18(weights=None)` initialises randomly — you lose ImageNet features and must train from scratch. Always pass `weights="DEFAULT"` or the specific weight enum to load pretrained weights.

**2. Not resizing inputs to the model's expected resolution**  
ResNet expects 224×224; EfficientNet-B3 expects 300×300; ViT-B/16 requires inputs in multiples of the patch size (16×16). Passing the wrong resolution may not raise an error (due to adaptive pooling) but degrades performance because the model was optimised for a specific resolution during pretraining.

**3. Feature extraction without `model.eval()` and `torch.no_grad()`**  
In train mode, BatchNorm updates running statistics from the feature extraction batch, corrupting the model's accumulated statistics. Always call `model.eval()` before feature extraction. `torch.no_grad()` prevents unnecessary memory allocation for gradients.

**4. Comparing model accuracy without controlling for training budget**  
A larger model trained for 10 epochs may underperform a smaller model trained for 50 epochs. Always train all compared models to convergence (monitor validation loss plateau) before comparing accuracy, and report the number of training epochs alongside accuracy.

**5. Deploying without testing inference on target hardware**  
Benchmarks on a development GPU do not reflect latency on the deployment device (CPU server, edge device, mobile). Always measure latency on the actual deployment hardware and test with the production batch size. A model that runs at 100ms on a GPU may run at 2000ms on CPU.
---
*python_methods_library - Samantha McGarrigle*